# MagiCoder LoRA Fine-Tuning — OSS-Instruct-75K (Python)

Fine-tunes LoRA adapters on the three PT base models using **ise-uiuc/Magicoder-OSS-Instruct-75K** (Python-only subset).

## Why OSS-Instruct over Evol-Instruct
- Sourced from real open-source code → higher quality and diversity
- `lang` column allows clean Python-only filtering
- Lower benchmark contamination risk (Evol-Instruct is evolved from HumanEval-style problems)
- Code-completion style better matches PT model pretraining

## Key fixes vs. previous MagiCoder training
| Issue | Previous | Fixed |
|---|---|---|
| 270m learning rate | `2e-4` (same as 4b) | `5e-5` (size-appropriate) |
| 270m LoRA scale | `alpha=32, r=16 → ×2.0` | `alpha=16, r=16 → ×1.0` |
| Warmup | 300 steps (same for all) | 500 steps for 270m, 300 for others |

These caused the 270m checkpoints to peak at step ~1000 then collapse to 0%.

## Outputs
- Adapters → `lora_outputs/lora_gemma-3-{size}_magicoder`
- Checkpoints → `checkpoints/ckpt_gemma-3-{size}_magicoder/checkpoint-*`

In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import torch, transformers, datasets, peft, trl
print(f"PyTorch:      {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
print(f"Transformers: {transformers.__version__}")
print(f"Datasets:     {datasets.__version__}")
print(f"PEFT:         {peft.__version__}")
print(f"TRL:          {trl.__version__}")
if torch.cuda.is_available():
    print(f"GPU:          {torch.cuda.get_device_name(0)}")
    print(f"VRAM:         {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

k:\anaconda3\envs\honor\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch:      2.11.0.dev20260205+cu128  |  CUDA: True
Transformers: 4.57.6
Datasets:     4.5.0
PEFT:         0.18.1
TRL:          0.27.2
GPU:          NVIDIA GeForce RTX 5070 Ti
VRAM:         15.9 GB


## 1. Load & Save Dataset

Loads `ise-uiuc/Magicoder-OSS-Instruct-75K`, filters to Python only, formats as `### User / ### Assistant`, and saves to disk so all three model training runs use identical data.

In [2]:
from datasets import load_dataset, Dataset, load_from_disk

DATA_SAVE_PATH = r"K:\Honor Project\datasets\magicoder_oss_python"

if os.path.isdir(DATA_SAVE_PATH):
    print(f"Loading saved dataset from {DATA_SAVE_PATH}...")
    magicoder_ds = load_from_disk(DATA_SAVE_PATH)
    print(f"  Loaded: {len(magicoder_ds):,} examples")
else:
    print("Downloading ise-uiuc/Magicoder-OSS-Instruct-75K...")
    _raw = load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K", split="train")
    print(f"  Total rows: {len(_raw):,}")
    print(f"  Columns: {_raw.column_names}")

    # Language distribution
    from collections import Counter
    lang_counts = Counter(_raw["lang"])
    print("\n  Language distribution:")
    for lang, cnt in sorted(lang_counts.items(), key=lambda x: -x[1]):
        print(f"    {lang:<20} {cnt:>6,}")

    # Filter to Python only
    _py = _raw.filter(lambda x: x["lang"].lower() == "python", num_proc=4)
    print(f"\n  Python-only rows: {len(_py):,}")

    # Format as ### User / ### Assistant  (matches PT model training format)
    def format_example(ex):
        text = (
            f"### User:\n{ex['problem']}\n"
            f"### Assistant:\n{ex['solution']}"
        )
        return {"text": text}

    _ds = _py.map(format_example, num_proc=4, remove_columns=_py.column_names)

    print(f"  Saving to {DATA_SAVE_PATH}...")
    os.makedirs(os.path.dirname(DATA_SAVE_PATH), exist_ok=True)
    _ds.save_to_disk(DATA_SAVE_PATH)
    magicoder_ds = _ds
    print(f"  Saved {len(magicoder_ds):,} examples.")

print(f"\nDataset ready: {len(magicoder_ds):,} examples")
print(f"Sample:\n{magicoder_ds[0]['text'][:500]}...")

k:\anaconda3\envs\honor\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\dongs\.cache\huggingface\hub\datasets--ise-uiuc--Magicoder-OSS-Instruct-75K. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 75197/75197 [00:00<00:00, 229197.62 examples/s]


  Total rows: 75,197
  Columns: ['lang', 'raw_index', 'index', 'seed', 'openai_fingerprint', 'problem', 'solution']

  Language distribution:
    python               38,284
    shell                 4,730
    typescript            4,700
    cpp                   4,699
    rust                  4,695
    php                   4,576
    java                  4,565
    swift                 4,498
    csharp                4,450


Filter (num_proc=4): 100%|██████████| 75197/75197 [00:06<00:00, 12277.28 examples/s]



  Python-only rows: 38,284


Map (num_proc=4): 100%|██████████| 38284/38284 [00:06<00:00, 6019.33 examples/s] 


  Saving to K:\Honor Project\datasets\magicoder_oss_python...


Saving the dataset (1/1 shards): 100%|██████████| 38284/38284 [00:00<00:00, 588360.41 examples/s]

  Saved 38,284 examples.

Dataset ready: 38,284 examples
Sample:
### User:
You are tasked with implementing a simple Python function that takes a list of strings as input and returns a new list containing only the strings that are palindromes. A palindrome is a word, phrase, number, or other sequence of characters that reads the same forward and backward (ignoring spaces, punctuation, and capitalization).

You are provided with the following code snippet as a starting point:

```python
def find_palindromes(words):
    # Your code here
    return palindromes
`...


## 2. Training Configuration

### Per-model hyperparameters (key fix)

| Model | LR | LoRA alpha | Scale (α/r) | Warmup |
|---|---|---|---|---|
| gemma-3-270m | `5e-5` | 16 | ×1.0 | 500 steps |
| gemma-3-1b-pt | `1e-4` | 32 | ×2.0 | 300 steps |
| gemma-3-4b-pt | `5e-5` | 32 | ×2.0 | 300 steps |

The 270m model previously used `lr=2e-4` with `alpha=32` (effective LR = `4e-4`) — too aggressive for a 270M parameter model, causing training to collapse after step ~1000.

In [3]:
import re
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

HF_CACHE_ROOT = r"K:\hf_cache"
os.environ.setdefault("HF_HOME",            HF_CACHE_ROOT)
os.environ.setdefault("HF_DATASETS_CACHE",  os.path.join(HF_CACHE_ROOT, "datasets"))
os.environ.setdefault("TRANSFORMERS_CACHE", os.path.join(HF_CACHE_ROOT, "hub"))
os.environ.setdefault("TMP",                os.path.join(HF_CACHE_ROOT, "tmp"))
os.environ.setdefault("TEMP",               os.path.join(HF_CACHE_ROOT, "tmp"))

CFG = {
    # ── Models to train (PT base models only) ────────────────────────────────
    "model_dirs": [
        r"K:\Honor Project\hf_models\gemma-3-270m",
        r"K:\Honor Project\hf_models\gemma-3-1b-pt",
        r"K:\Honor Project\hf_models\gemma-3-4b-pt",
    ],
    "output_base": r"K:\Honor Project\lora_outputs",
    "ckpt_base":   r"K:\Honor Project\checkpoints",
    "dataset_tag": "magicoder",

    # ── Training budget ───────────────────────────────────────────────────────
    "max_length":  1024,
    "epochs":      1,
    "max_steps":   10000,
    "bsz":         1,
    "gas":         16,       # effective batch = 16

    # ── Per-model learning rates ──────────────────────────────────────────────
    # FIX: 270m previously used same lr=2e-4 as larger models → training collapse
    # Rule of thumb: smaller model → lower LR to prevent divergence
    "lr_by_model": {
        "gemma-3-270m":  5e-5,   # conservative — 270m has limited capacity
        "gemma-3-1b-pt": 1e-4,   # moderate
        "gemma-3-4b-pt": 5e-5,   # conservative for large model stability
    },
    "lr": 5e-5,   # fallback

    # ── Per-model LoRA alpha ──────────────────────────────────────────────────
    # scale = alpha / r.  FIX: 270m was alpha=32 (scale=2.0) → effective LR ×2
    "lora_alpha_by_model": {
        "gemma-3-270m":  16,   # scale = 16/16 = 1.0  (neutral — no amplification)
        "gemma-3-1b-pt": 32,  # scale = 32/16 = 2.0
        "gemma-3-4b-pt": 32,  # scale = 32/16 = 2.0
    },
    "lora_alpha": 16,   # fallback

    # ── Per-model warmup ──────────────────────────────────────────────────────
    # FIX: 270m needs longer warmup to stabilise early-phase updates
    "warmup_by_model": {
        "gemma-3-270m":  500,   # 5% of 10k steps
        "gemma-3-1b-pt": 300,   # 3%
        "gemma-3-4b-pt": 300,   # 3%
    },
    "warmup_steps": 300,   # fallback

    # ── Other settings ────────────────────────────────────────────────────────
    "lora_r":       16,
    "lora_dropout": 0.05,
    "save_every":   500,
    "eval_ratio":   0.01,
    "seed":         42,
}


def safe_tag(model_dir):
    return re.sub(r"[^a-zA-Z0-9._-]+", "_", os.path.basename(model_dir.rstrip("\\/")))


def get_hparam(model_tag, key, fallback_key):
    """Look up per-model hyperparameter by substring match on model tag."""
    tag_norm = model_tag.replace("_", "-").lower()
    for model_key, val in CFG[key].items():
        if model_key.lower() in tag_norm:
            return val
    return CFG[fallback_key]


print("Configuration summary:")
print(f"  Dataset:    ise-uiuc/Magicoder-OSS-Instruct-75K (Python only)")
print(f"  Saved at:   {DATA_SAVE_PATH}")
print(f"  Examples:   {len(magicoder_ds):,}")
print(f"  max_steps:  {CFG['max_steps']}  |  bsz={CFG['bsz']}  gas={CFG['gas']}  eff_bsz={CFG['bsz']*CFG['gas']}")
print()
print(f"{'Model':<20}  {'LR':>8}  {'alpha':>6}  {'scale':>6}  {'warmup':>7}")
print("-" * 60)
for mdir in CFG["model_dirs"]:
    tag = safe_tag(mdir)
    lr  = get_hparam(tag, "lr_by_model",      "lr")
    alp = get_hparam(tag, "lora_alpha_by_model", "lora_alpha")
    wu  = get_hparam(tag, "warmup_by_model",   "warmup_steps")
    print(f"{tag:<20}  {lr:>8.0e}  {alp:>6}  {alp/CFG['lora_r']:>6.1f}  {wu:>7}")

Configuration summary:
  Dataset:    ise-uiuc/Magicoder-OSS-Instruct-75K (Python only)
  Saved at:   K:\Honor Project\datasets\magicoder_oss_python
  Examples:   38,284
  max_steps:  10000  |  bsz=1  gas=16  eff_bsz=16

Model                       LR   alpha   scale   warmup
------------------------------------------------------------
gemma-3-270m             5e-05      16     1.0      500
gemma-3-1b-pt            1e-04      32     2.0      300
gemma-3-4b-pt            5e-05      32     2.0      300


## 3. Train All Three Models

Loads each base model CPU-first (RTX 5070 Ti / SM 12.0 workaround), applies LoRA, trains for 10,000 steps, saves checkpoints every 500 steps and the final adapter.

In [4]:
os.makedirs(CFG["output_base"], exist_ok=True)
os.makedirs(CFG["ckpt_base"],   exist_ok=True)

torch.manual_seed(CFG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CFG["seed"])

# ── Train/eval split (fixed across all models — same data for fair comparison) ─
_split    = magicoder_ds.train_test_split(test_size=CFG["eval_ratio"], seed=CFG["seed"])
ds_train  = _split["train"]
ds_eval   = _split["test"]
print(f"Train: {len(ds_train):,}  |  Eval: {len(ds_eval):,}")
print()

# ── GPU dtype detection ───────────────────────────────────────────────────────
_sm = torch.cuda.get_device_capability(0)[0] if torch.cuda.is_available() else 0
DTYPE   = torch.bfloat16 if _sm >= 8 else torch.float16
USE_BF16 = _sm >= 8
USE_FP16 = torch.cuda.is_available() and not USE_BF16
print(f"Dtype: {DTYPE}  (bf16={USE_BF16}, fp16={USE_FP16})")
print()

# ── Training loop ─────────────────────────────────────────────────────────────
for model_dir in CFG["model_dirs"]:
    tag         = safe_tag(model_dir)
    model_lr    = get_hparam(tag, "lr_by_model",         "lr")
    model_alpha = get_hparam(tag, "lora_alpha_by_model", "lora_alpha")
    model_wu    = get_hparam(tag, "warmup_by_model",     "warmup_steps")

    final_dir = os.path.join(CFG["output_base"], f"lora_{tag}_{CFG['dataset_tag']}")
    ckpt_dir  = os.path.join(CFG["ckpt_base"],   f"ckpt_{tag}_{CFG['dataset_tag']}")
    os.makedirs(final_dir, exist_ok=True)
    os.makedirs(ckpt_dir,  exist_ok=True)

    print("=" * 80)
    print(f"Model:      {model_dir}")
    print(f"Adapter  -> {final_dir}")
    print(f"Checkpts -> {ckpt_dir}")
    print(f"lr={model_lr:.0e}  alpha={model_alpha} (scale={model_alpha/CFG['lora_r']:.1f})  warmup={model_wu}")
    print("=" * 80)

    # ── Load tokenizer ────────────────────────────────────────────────────────
    tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "right"

    # ── Load model CPU-first (SM 12.0 / RTX 5070 Ti workaround) ──────────────
    print("  Loading model on CPU...")
    model = AutoModelForCausalLM.from_pretrained(
        model_dir,
        local_files_only=True,
        device_map=None,
        dtype=DTYPE,
        attn_implementation="sdpa",
    )
    model.config.use_cache = False
    if torch.cuda.is_available():
        print("  Moving to GPU...")
        model = model.to("cuda")

    # ── LoRA config ───────────────────────────────────────────────────────────
    peft_config = LoraConfig(
        r=CFG["lora_r"],
        lora_alpha=model_alpha,
        lora_dropout=CFG["lora_dropout"],
        bias="none",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )

    # ── SFT config ────────────────────────────────────────────────────────────
    sft_args = SFTConfig(
        output_dir=ckpt_dir,
        max_length=CFG["max_length"],
        packing=False,
        dataset_text_field="text",
        dataloader_num_workers=0,
        neftune_noise_alpha=None,

        num_train_epochs=CFG["epochs"],
        max_steps=CFG["max_steps"],
        per_device_train_batch_size=CFG["bsz"],
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=CFG["gas"],

        learning_rate=model_lr,
        warmup_steps=model_wu,
        lr_scheduler_type="cosine",

        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        optim="adamw_torch_fused",

        fp16=USE_FP16,
        bf16=USE_BF16,
        tf32=torch.cuda.is_available(),
        dataloader_pin_memory=False,

        logging_steps=10,
        save_strategy="steps",
        save_steps=CFG["save_every"],
        save_total_limit=None,   # keep all checkpoints
        eval_strategy="steps",
        eval_steps=CFG["save_every"],
        report_to="none",
        seed=CFG["seed"],
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_args,
        train_dataset=ds_train,
        eval_dataset=ds_eval,
        peft_config=peft_config,
        processing_class=tok,
    )

    print("  Starting training...")
    trainer.train()
    trainer.save_model(final_dir)
    print(f"  Adapter saved -> {final_dir}")
    print(f"  Checkpoints   -> {ckpt_dir}")

    del model, trainer
    torch.cuda.empty_cache()
    print()

print("All models trained.")

Train: 37,901  |  Eval: 383

Dtype: torch.bfloat16  (bf16=True, fp16=False)

Model:      K:\Honor Project\hf_models\gemma-3-270m
Adapter  -> K:\Honor Project\lora_outputs\lora_gemma-3-270m_magicoder
Checkpts -> K:\Honor Project\checkpoints\ckpt_gemma-3-270m_magicoder
lr=5e-05  alpha=16 (scale=1.0)  warmup=500
  Loading model on CPU...
  Moving to GPU...


Truncating eval dataset: 100%|██████████| 383/383 [00:00<00:00, 23185.66 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


  Starting training...


Step,Training Loss,Validation Loss
500,0.997000,0.996139
1000,0.955800,0.950426
1500,0.910600,0.931388
2000,0.935600,0.917556
2500,0.908800,0.907007
3000,0.899900,0.900047
3500,0.879200,0.893236
4000,0.895200,0.887555
4500,0.854600,0.882730
5000,0.848500,0.879626


  Adapter saved -> K:\Honor Project\lora_outputs\lora_gemma-3-270m_magicoder
  Checkpoints   -> K:\Honor Project\checkpoints\ckpt_gemma-3-270m_magicoder

Model:      K:\Honor Project\hf_models\gemma-3-1b-pt
Adapter  -> K:\Honor Project\lora_outputs\lora_gemma-3-1b-pt_magicoder
Checkpts -> K:\Honor Project\checkpoints\ckpt_gemma-3-1b-pt_magicoder
lr=1e-04  alpha=32 (scale=2.0)  warmup=300
  Loading model on CPU...
  Moving to GPU...


...

  Moving to GPU...


Truncating eval dataset: 100%|██████████| 383/383 [00:00<00:00, 28028.87 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 0}.


  Starting training...


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

## 4. Verify Outputs

In [ ]:
print("Saved adapters:")
for model_dir in CFG["model_dirs"]:
    tag       = safe_tag(model_dir)
    final_dir = os.path.join(CFG["output_base"], f"lora_{tag}_{CFG['dataset_tag']}")
    ckpt_dir  = os.path.join(CFG["ckpt_base"],   f"ckpt_{tag}_{CFG['dataset_tag']}")

    adapter_ok = os.path.exists(os.path.join(final_dir, "adapter_config.json"))
    ckpts = sorted(
        [d for d in os.listdir(ckpt_dir) if d.startswith("checkpoint-")],
        key=lambda x: int(x.split("-")[1])
    ) if os.path.isdir(ckpt_dir) else []

    print(f"  {tag}")
    print(f"    Adapter: {'OK' if adapter_ok else 'MISSING'} ({final_dir})")
    print(f"    Checkpoints: {len(ckpts)} saved  [{ckpts[0] if ckpts else '—'} .. {ckpts[-1] if ckpts else '—'}]")
    print()